In [1]:
%%file app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")                      # URL: http://localhost:5000/
def home():
    return "Welcome to the transaction monitoring system!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing app.py


In [2]:
import subprocess, time, requests

server = subprocess.Popen(["python", "app.py"])
time.sleep(2)   # give the server a moment to start

response = requests.get("http://localhost:5000/")
print(f"Status: {response.status_code}")
print(f"Body:   {response.text}")

Status: 200
Body:   Welcome to the transaction monitoring system!


In [3]:
server.kill()

In [4]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")                  # GET /hello?name=Anna
def hello():
    name = request.args.get("name", "stranger")   # read query parameter
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>")    # GET /transaction/TX0042
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [5]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

# Query parameter
r1 = requests.get("http://localhost:5000/hello")
r2 = requests.get("http://localhost:5000/hello?name=Anna")
print(r1.text)   # Hello, stranger!
print(r2.text)   # Hello, Anna!

# Path variable
r3 = requests.get("http://localhost:5000/transaction/TX0042")
print(r3.json())

Hello, stranger!
Hello, Anna!
{'status': 'found', 'tx_id': 'TX0042'}


In [6]:
server.kill()

In [7]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")               # GET /hello?name=Anna
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>") # GET /transaction/TX0042
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})

# Task 2.2 — Status endpoint
@app.route("/status")
def status():
    return jsonify({
        "service": "monitoring",
        "version": "1.0",
        "status": "ok"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [8]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

# Query parameter
r1 = requests.get("http://localhost:5000/hello")
r2 = requests.get("http://localhost:5000/hello?name=Anna")
print(r1.text)
print(r2.text)

# Path variable
r3 = requests.get("http://localhost:5000/transaction/TX0042")
print(r3.json())

# Task 2.2 — Status endpoint
r4 = requests.get("http://localhost:5000/status")
print(r4.json())

Hello, stranger!
Hello, Anna!
{'status': 'found', 'tx_id': 'TX0042'}
{'service': 'monitoring', 'status': 'ok', 'version': '1.0'}


In [9]:
server.kill()

In [10]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/echo", methods=["POST"])
def echo():
    data = request.get_json()
    return jsonify({
        "received": data,
        "field_count": len(data),
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [11]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

transaction = {
    "tx_id": "TX0042",
    "amount": 4500.0,
    "store": "Krakow",
    "category": "electronics",
}

r = requests.post("http://localhost:5000/echo", json=transaction)
print(r.json())

{'field_count': 4, 'received': {'amount': 4500.0, 'category': 'electronics', 'store': 'Krakow', 'tx_id': 'TX0042'}}


In [12]:
r_bad = requests.get("http://localhost:5000/echo")
print(f"Status: {r_bad.status_code}")

# ANSWER:
"""

Status 405 = "Method Not Allowed". The endpoint /echo is registered with methods=["POST"] only, 
so calling it with GET is rejected. The server knows the URL exists but not for this HTTP method.

"""

Status: 405


In [13]:
server.kill()

In [14]:
def score_transaction(tx: dict) -> dict:
    """
    Assess transaction risk using business rules.
    Returns dict with: score, risk_level, triggered_rules.
    """
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")

    # Rule R3: night-hour transactions (hour < 6)
    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}


# Quick test
test_tx = {"tx_id": "TX001", "amount": 4500.0, "category": "electronics", "hour": 3}
print(score_transaction(test_tx))

{'score': 7, 'risk_level': 'CRITICAL', 'triggered_rules': ['R1: amount > 3000', 'R2: electronics > 1500', 'R3: night hour']}


In [15]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

def score_transaction(tx):
    score = 0
    rules = []
    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: amount > 3000")
    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: electronics > 1500")
    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: night hour")
    risk_level = (
        "CRITICAL" if score >= 5
        else "HIGH" if score >= 3
        else "MEDIUM" if score >= 1
        else "LOW"
    )
    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()
    if not tx or "amount" not in tx:
        return jsonify({"error": "Missing required field 'amount'"}), 400
    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")
    return jsonify(result)

@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.0-rules"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [16]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

cases = [
    {"tx_id": "TX001", "amount": 50.0,   "category": "food",        "hour": 14},
    {"tx_id": "TX002", "amount": 1800.0, "category": "electronics", "hour": 10},
    {"tx_id": "TX003", "amount": 4500.0, "category": "electronics", "hour": 3},
]

for tx in cases:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    print(f"{tx['tx_id']}  {tx['amount']:>7.0f} PLN  →  {res['risk_level']:8s} (score={res['score']})  {res['triggered_rules']}")

TX001       50 PLN  →  LOW      (score=0)  []
TX002     1800 PLN  →  MEDIUM   (score=2)  ['R2: electronics > 1500']
TX003     4500 PLN  →  CRITICAL (score=7)  ['R1: amount > 3000', 'R2: electronics > 1500', 'R3: night hour']


In [17]:
r = requests.post("http://localhost:5000/score", json={"tx_id": "TX000"})
print(f"Status: {r.status_code}")
print(f"Body:   {r.json()}")

Status: 400
Body:   {'error': "Missing required field 'amount'"}


In [18]:
server.kill()

## Homework

In [19]:
%%file app.py
from flask import Flask, request, jsonify
from threading import Lock

app = Flask(__name__)

# ---- HW 1: shared counters with thread-safe lock ----
counters = {"total": 0, "high": 0, "critical": 0}
counters_lock = Lock()

def score_transaction(tx):
    score = 0
    rules = []
    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: amount > 3000")
    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: electronics > 1500")
    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: night hour")
    risk_level = (
        "CRITICAL" if score >= 5
        else "HIGH" if score >= 3
        else "MEDIUM" if score >= 1
        else "LOW"
    )
    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()

    # HW 2 — validation: missing amount
    if not tx or "amount" not in tx:
        return jsonify({"error": "Missing required field 'amount'"}), 400

    # HW 2 — validation: negative amount
    if tx["amount"] < 0:
        return jsonify({"error": "Field 'amount' cannot be negative"}), 400

    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")

    # HW 1 — update counters (thread-safe)
    with counters_lock:
        counters["total"] += 1
        if result["risk_level"] == "HIGH":
            counters["high"] += 1
        elif result["risk_level"] == "CRITICAL":
            counters["critical"] += 1

    return jsonify(result)

# HW 1 — /stats endpoint
@app.route("/stats")
def stats():
    with counters_lock:
        return jsonify({
            "total_requests": counters["total"],
            "high_alerts":    counters["high"],
            "critical_alerts": counters["critical"],
        })

@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.1-homework"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [20]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

# --- HW 2: test validation ---
print("=== HW 2: Validation ===")

# Test 1: missing amount
r = requests.post("http://localhost:5000/score", json={"tx_id": "TX000"})
print(f"Missing amount:   Status={r.status_code}  Body={r.json()}")

# Test 2: negative amount
r = requests.post("http://localhost:5000/score",
                  json={"tx_id": "TX_NEG", "amount": -100.0})
print(f"Negative amount:  Status={r.status_code}  Body={r.json()}")

# Test 3: valid request
r = requests.post("http://localhost:5000/score",
                  json={"tx_id": "TX_OK", "amount": 500.0, "category": "food", "hour": 14})
print(f"Valid request:    Status={r.status_code}  Body={r.json()}")

=== HW 2: Validation ===
Missing amount:   Status=400  Body={'error': "Missing required field 'amount'"}
Negative amount:  Status=400  Body={'error': "Field 'amount' cannot be negative"}
Valid request:    Status=200  Body={'risk_level': 'LOW', 'score': 0, 'triggered_rules': [], 'tx_id': 'TX_OK'}


In [21]:
import random
random.seed(42)

stores = ["Warsaw", "Krakow", "Gdansk", "Wroclaw"]
categories = ["electronics", "clothing", "food", "books"]

transactions = []
for i in range(10):
    transactions.append({
        "tx_id":    f"TX{i+1:03d}",
        "amount":   round(random.uniform(50, 5000), 2),
        "category": random.choice(categories),
        "store":    random.choice(stores),
        "hour":     random.randint(0, 23),
    })

print(f"\n{'='*85}")
print(f"{'tx_id':<8} {'amount':>10} {'category':<13} {'hour':>5} {'risk':<10} {'score':>6}  rules")
print("-" * 85)

risk_summary = {"LOW": 0, "MEDIUM": 0, "HIGH": 0, "CRITICAL": 0}

for tx in transactions:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    risk_summary[res["risk_level"]] += 1
    rules_str = ", ".join(r.split(":")[0] for r in res["triggered_rules"]) or "—"
    print(f"{tx['tx_id']:<8} {tx['amount']:>10.2f} {tx['category']:<13} "
          f"{tx['hour']:>5} {res['risk_level']:<10} {res['score']:>6}  {rules_str}")

print("=" * 85)
print(f"\nRisk summary: {risk_summary}")

# Verify /stats endpoint (HW 1)
stats = requests.get("http://localhost:5000/stats").json()
print(f"\nServer-side /stats: {stats}")


tx_id        amount category       hour risk        score  rules
-------------------------------------------------------------------------------------
TX001       3215.16 electronics       7 CRITICAL        5  R1, R2
TX002       1154.89 electronics      18 LOW             0  —
TX003       2138.51 electronics       6 MEDIUM          2  R2
TX004       1201.67 electronics      22 LOW             0  —
TX005       3266.93 books            14 HIGH            3  R1
TX006       2966.87 electronics      22 MEDIUM          2  R2
TX007       2141.94 food              6 LOW             0  —
TX008       4788.20 food              2 CRITICAL        5  R1, R3
TX009       1930.64 food             19 LOW             0  —
TX010       1359.38 electronics      17 LOW             0  —

Risk summary: {'LOW': 5, 'MEDIUM': 2, 'HIGH': 1, 'CRITICAL': 2}

Server-side /stats: {'critical_alerts': 2, 'high_alerts': 1, 'total_requests': 11}


In [22]:
server.kill()